In [98]:
import pandas as pd
import numpy as np

In [99]:
df = pd.read_csv('dataset/dataset.csv')

In [100]:
df.head(2)

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin


In [101]:
def tokenize_text(text):
    text = text.replace("?", "").replace("'", "")
    return text.split()

In [102]:
## example of tokenization
text = df.head(1).iloc[0]
text['question']

tokenize_text(text=text['question'])

['What', 'is', 'the', 'capital', 'of', 'France']

In [103]:
voc = {'<UNK>' :0}

In [104]:
i = 1
def build_voc(row):
    global i 
    sentences = tokenize_text(row['question']) + tokenize_text(row['answer'])
    
    for sentence in sentences:
        if sentence not in voc:
            voc[sentence] = i
            i+=1
    

In [105]:
df.apply(build_voc,axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [106]:
voc

{'<UNK>': 0,
 'What': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'France': 6,
 'Paris': 7,
 'Germany': 8,
 'Berlin': 9,
 'Who': 10,
 'wrote': 11,
 'To': 12,
 'Kill': 13,
 'a': 14,
 'Mockingbird': 15,
 'Harper-Lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'Jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'Celsius': 27,
 '100': 28,
 'painted': 29,
 'Mona': 30,
 'Lisa': 31,
 'Leonardo-da-Vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'Au': 41,
 'Which': 42,
 'year': 43,
 'did': 44,
 'World': 45,
 'War': 46,
 'II': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'world': 52,
 'Nile': 53,
 'Japan': 54,
 'Tokyo': 55,
 'developed': 56,
 'theory': 57,
 'relativity': 58,
 'Albert-Einstein': 59,
 'freezing': 60,
 'Fahrenheit': 61,
 '32': 62,
 'known': 63,
 'as': 64,
 'Red': 65,
 'Planet': 66,
 'Mars': 67,
 'author': 68,
 '1984': 69,
 'George-Orwell'

In [107]:
def index_text(text):
    indexes = []
    for t in text:
        if t in voc:
            indexes.append(voc[t])
        else:
            indexes.append(voc['<UNK>'])
    return indexes

In [108]:
index_text(tokenize_text(text['question']))

[1, 2, 3, 4, 5, 6]

In [109]:
import torch
from torch.utils.data import Dataset, DataLoader

In [110]:
class CustomDataset(Dataset):
    def __init__(self, df):
        self.df = df
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        question = self.df.iloc[idx]['question']
        answer = self.df.iloc[idx]['answer']
        
        question_indexes = index_text(tokenize_text(question))
        answer_indexes = index_text(tokenize_text(answer))
        
        return torch.tensor(question_indexes), torch.tensor(answer_indexes)

In [111]:
dataset = CustomDataset(df)

In [112]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [113]:
for questions,answers in dataloader:
    print(questions)
    print(answers)
    break

tensor([[ 80,  81, 266, 156,  14, 267, 158]])
tensor([[36]])


In [114]:
import torch.nn as nn

In [115]:
class NN(nn.Module):
    def __init__(self):
        super(NN, self).__init__()
        self.embedding = nn.Embedding(len(voc), 50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, len(voc))
        
    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))

        return output

In [116]:
model = NN()

In [117]:
EPOCHS = 50
learning_rate = 0.001

In [118]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [119]:
for epoch in range(EPOCHS):
    total_loss = 0
    for questions, answers in dataloader:
        optimizer.zero_grad()
        output = model(questions)
        loss = criterion(output, answers.squeeze(1))
        loss.backward()
        optimizer.step()
        total_loss = total_loss + loss.item()

    print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 525.162955
Epoch: 2, Loss: 456.546781
Epoch: 3, Loss: 377.684983
Epoch: 4, Loss: 317.223524
Epoch: 5, Loss: 266.590902
Epoch: 6, Loss: 219.116906
Epoch: 7, Loss: 175.880915
Epoch: 8, Loss: 137.712160
Epoch: 9, Loss: 106.066425
Epoch: 10, Loss: 81.394347
Epoch: 11, Loss: 62.280793
Epoch: 12, Loss: 48.593490
Epoch: 13, Loss: 38.648100
Epoch: 14, Loss: 30.999303
Epoch: 15, Loss: 25.451626
Epoch: 16, Loss: 20.711863
Epoch: 17, Loss: 17.672202
Epoch: 18, Loss: 14.945296
Epoch: 19, Loss: 12.774676
Epoch: 20, Loss: 10.999902
Epoch: 21, Loss: 9.604615
Epoch: 22, Loss: 8.448274
Epoch: 23, Loss: 7.455956
Epoch: 24, Loss: 6.663461
Epoch: 25, Loss: 5.973812
Epoch: 26, Loss: 5.379063
Epoch: 27, Loss: 4.866602
Epoch: 28, Loss: 4.416402
Epoch: 29, Loss: 4.030461
Epoch: 30, Loss: 3.691064
Epoch: 31, Loss: 3.387687
Epoch: 32, Loss: 3.113438
Epoch: 33, Loss: 2.878190
Epoch: 34, Loss: 2.668417
Epoch: 35, Loss: 2.464705
Epoch: 36, Loss: 2.287403
Epoch: 37, Loss: 2.133804
Epoch: 38, Loss: 1

In [136]:
def predict(model, question, threshold=0.5):
    model.eval()

    # tokenize + convert question to numbers
    numerical_question = index_text(tokenize_text(question))

    # tensor
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)

    with torch.no_grad():
        # send to model
        output = model(question_tensor)

        # convert logits to probs
        probs = torch.nn.functional.softmax(output, dim=1)

        # find index of max prob
        value, index = torch.max(probs, dim=1)

    confidence = value.item()
    predicted_index = index.item()

    if confidence < threshold:
        return "I don't know"

    predicted_token = list(voc.keys())[predicted_index]
 
    return predicted_token

In [138]:
answer = predict(model, "capital of France?")
answer

'Paris'